# 03-1. 문서 전처리와 Chunking

- 예상 시간: 90분 (선택·심화 포함 시 120분)
- 선수 실습: 02-3_langgraph_tool_agent.ipynb (RAG 트랙만 별도로 수강하면 생략 가능)
- 실습 난이도: 기초~중급
- 핵심 기술: 고정 길이, Overlap, 구조 기반 Chunking, `RecursiveCharacterTextSplitter`
- 최종 산출물: Agentic RAG에 사용할 표준 Chunk 목록과 전략 비교 결과
- 버전: 학생용 완성본 (모든 코드가 완성되어 있습니다. 직접 실행해 결과를 확인하세요)

## 1. 학습목표

이 실습을 완료하면 다음을 수행할 수 있다.

1. 문서 전처리와 Chunking이 왜 필요한지 설명할 수 있다.
2. 고정 길이, Overlap, 구조 기반 Chunking의 차이를 비교할 수 있다.
3. Chunk Size와 Overlap이 문맥 손실, 주제 혼합, 검색 정밀도에 미치는 영향을 설명할 수 있다.
4. 실제 RAG에서는 `RecursiveCharacterTextSplitter`를 사용해 Chunk를 생성할 수 있다.
5. 선택·심화 학습으로 문장/문단 직접 구현과 Parent-Child 구조의 용도를 설명할 수 있다.

## 2. 문제 상황

`data/samples/sample_report.txt`처럼 여러 섹션으로 구성된 보고서를 검색 기반 Agent에
활용하려면, 문서 전체를 통째로 LLM에 넘길 수 없다. LLM이 한 번에 참고할 수 있는 입력
길이는 제한되어 있고, 문서 전체를 검색 후보로 쓰면 질문과 무관한 내용까지 함께
반환되어 정확도와 비용이 나빠진다. 문서를 어떤 기준으로, 얼마나 작게 나누는지가
이후 모든 검색(Notebook 03-2)과 Agentic RAG(Notebook 03-3) 품질을 좌우한다. 이 Notebook은
알고리즘 구현보다 Chunk 크기·Overlap·문서 구조의 영향을 판단하고, 실제 RAG에서
사용할 Text Splitter를 적용하는 데 집중한다.

## 3. 핵심 개념

### 3.1 정의

Chunking은 긴 문서를 검색과 LLM 입력에 적합한 크기의 작은 조각(Chunk)으로 나누는
전처리 과정이다. 각 Chunk는 독립적으로 검색되고 LLM에 전달될 수 있는 최소 단위가
된다.

### 3.2 개념이 필요한 이유

LLM의 Context 길이는 제한되어 있고, 검색 시스템은 "문서" 단위가 아니라 "Chunk" 단위로
후보를 찾는다. 문서를 나누지 않으면 질문과 관련된 한두 문장을 찾기 위해 문서 전체를
후보로 반환해야 하므로 정확도가 낮고 비용이 커진다. 반대로 무작정 잘게 나누면 문맥이
끊겨 의미를 잃는다. Chunking은 이 균형점을 찾는 작업이다.

### 3.3 주요 구성요소

| 필드 | 의미 |
|---|---|
| `chunk_id` | Chunk 식별자 |
| `document_id` | 원본 문서 |
| `section` | 문서 위치(섹션 제목). 구조 정보를 모르는 전략은 `None` |
| `text` | Chunk 본문 |
| `length` | 글자 수 |
| `parent_id` | 상위 구조(부모 Chunk) 식별자. 없으면 `None` |

### 3.4 동작 과정

```text
문서
  ↓
전처리(공백/개행 정리)
  ↓
청킹 전략 판단 (고정 길이 / Overlap / 구조 기반)
  ↓
실제 RAG용 RecursiveCharacterTextSplitter 적용
  ↓
Chunk 목록 (chunk_id, document_id, section, text, length, parent_id)
```

### 3.5 코드와 개념의 대응 관계

| 코드 요소 | 구현 개념 |
|---|---|
| `chunk_by_sentence()` | 문장 단위 Chunking |
| `chunk_by_fixed_length()` | 고정 길이 Chunking |
| `chunk_by_fixed_length_with_overlap()` | Overlap Chunking |
| `chunk_by_paragraph()` | 문단 단위 Chunking |
| `chunk_by_structure()` | 제목과 본문을 함께 보존하는 섹션 Chunking |
| `RecursiveCharacterTextSplitter` | 실제 RAG용 재귀적 구분자 기반 Chunking |
| `build_parent_child_chunks()` [심화] | 상위 섹션과 하위 검색 Chunk 연결 |

### 3.6 유사 개념과의 차이

**작은 Chunk vs 큰 Chunk**

```text
Chunk가 너무 작음
→ 문맥 손실 및 관련 정보 분산

Chunk가 너무 큼
→ 여러 주제 혼합 및 검색 정밀도 저하

Overlap 증가
→ 문맥 단절 완화, 중복 저장과 검색 증가
```

**학습 범위와 전략 비교**

| 범위 | 전략 | 기준 | 학습 목적 |
|---|---|---|---|
| 필수 | 고정 길이 | 글자 수 | 작은/큰 Chunk의 문제 확인 |
| 필수 | Overlap | 글자 수 + 겹치는 구간 | 경계 문맥 손실과 중복 비용 비교 |
| 필수 | 구조 기반 | 제목/번호 패턴 | 제목-본문과 섹션 metadata 보존 |
| 필수 | `RecursiveCharacterTextSplitter` | 구분자 우선순위 + 길이 | 실제 Agent/RAG 코드에서 기존 기능 활용 |
| 선택 | 문장·문단 직접 구현 | 문장 부호·빈 줄 | 분리 원리와 한계 확인 |
| 심화 | Parent-Child | 상위 섹션 + 하위 Chunk | 검색 단위와 답변 맥락을 다르게 운영 |

### 3.7 사용 시점과 적용 조건

보고서, 매뉴얼처럼 제목과 번호로 구조가 뚜렷한 문서는 구조 기반 Chunking이 유리하다.
구조가 없는 대화 로그나 자유 형식 텍스트는 `RecursiveCharacterTextSplitter`에
적절한 구분자와 Overlap을 설정하는 방식이 현실적인 대안이다.

### 3.8 한계와 주의사항

- 고정 길이 Chunking은 글자 수만 보므로 단어나 문장 중간에서 잘릴 수 있다.
- Overlap을 늘리면 문맥 손실은 줄지만, 같은 내용이 여러 Chunk에 중복 저장되어
  검색 결과에도 중복이 늘어난다.
- 구조 기반 Chunking은 제목 패턴(`"숫자. 제목"` 등)이 문서마다 다르면 그대로
  적용되지 않는다.

### 3.9 자주 발생하는 오해

"Chunk를 작게 나눌수록 검색이 항상 더 정확해진다"는 오해가 있다. 실제로는 Chunk가
너무 작으면 문맥이 손실되어 관련 정보가 여러 조각으로 흩어지고, 오히려 검색과
답변 품질이 떨어질 수 있다. Chunk 크기는 "작을수록 좋다"가 아니라 문서 특성에 맞는
균형점을 찾는 문제다.

### 3.10 핵심 정리

- Chunking은 문서를 검색과 LLM 입력에 적합한 크기로 나누는 전처리 과정이다.
- Chunk가 너무 작으면 문맥 손실, 너무 크면 주제 혼합과 정밀도 저하가 발생한다.
- Overlap은 문맥 단절을 완화하지만 저장·검색 비용이 늘어나는 트레이드오프가 있다.
- 실제 Agentic RAG에서는 검증된 Text Splitter를 사용하고, 문서 특성에 맞게
  `chunk_size`, `chunk_overlap`, 구분자 우선순위를 조정한다.
- Parent-Child는 검색 단위와 상위 맥락을 분리해야 할 때 사용하는 심화 전략이다.

## 4. 실행 구조

```text
sample_report.txt 로드
   │
   ├─ [필수] chunk_by_fixed_length()
   ├─ [필수] chunk_by_fixed_length_with_overlap()
   ├─ [필수] chunk_by_structure()
   ├─ [필수] RecursiveCharacterTextSplitter
   ├─ [선택] chunk_by_sentence() / chunk_by_paragraph()
   └─ [심화] build_parent_child_chunks()
           │
           ↓
   전략별 비교 표 (chunk 수, 평균/최대/최소 길이)
           │
           ↓
   Chunk Size / Overlap / 제목-본문 분리 핵심 실험
           │
           ↓
   구조 기반 Chunking으로 문제 재검증
           │
           ↓
   결과 저장 (outputs/logs/03-1_document_chunking_log.json)
```

## 5. 환경 설정

반복되는 경로 탐색과 모델 생성 코드는 `src/agentic_ai` 공통 모듈에서 관리한다.
아래 셀에서는 이 Notebook에 필요한 표준 라이브러리와 공통 기능만 불러온다.

> 처음 실행하거나 환경 오류가 발생하면 프로젝트 루트의
> `00_environment_check.ipynb`를 먼저 실행한다.


In [21]:
import re

from langchain_text_splitters import RecursiveCharacterTextSplitter

from agentic_ai.logging_utils import save_log
from agentic_ai.notebook_utils import print_environment_summary
from agentic_ai.paths import OUTPUT_DIR, data_path
from agentic_ai.tools import load_document

DATA_PATH = data_path("sample_report.txt", must_exist=True)
print_environment_summary()

document_text = load_document(DATA_PATH)
print(f"문서 길이: {len(document_text)}자")
print(document_text[:80])


[환경 설정 확인]
- 프로젝트: E:\agentic_ai_lab
- 데이터: E:\agentic_ai_lab\data
- 출력: E:\agentic_ai_lab\outputs
문서 길이: 719자
2026년 사내 리모트워크 운영 현황 보고서

작성일: 2026-01-15
작성부서: 인사운영팀
문서구분: 내부 보고서

1. 개요
본 보고서는


## 6. 최소 실행 예제

전체 구현에 앞서, Chunking의 가장 단순한 형태를 확인한다. 마침표(`.`)로만 나누는
초간단 버전이다.

In [22]:
mini_text = "리모트워크 만족도는 74%로 높았다. 신입 온보딩 보완이 필요하다."
mini_chunks = [s.strip() for s in mini_text.split(".") if s.strip()]
for i, s in enumerate(mini_chunks, start=1):
    print(f"chunk-{i}: {s!r} (length={len(s)})")

chunk-1: '리모트워크 만족도는 74%로 높았다' (length=19)
chunk-2: '신입 온보딩 보완이 필요하다' (length=15)


## 7. 단계별 구현

### 7.1 공통 데이터 구조

모든 전략이 같은 형식의 Chunk 딕셔너리를 만들도록 `make_chunk()`를 공통으로
사용한다.

In [23]:
def make_chunk(chunk_id: str, document_id: str, section: str | None, text: str, parent_id: str | None = None) -> dict:
    """모든 Chunking 전략이 공유하는 표준 Chunk 형식을 만든다."""
    return {
        "chunk_id": chunk_id,
        "document_id": document_id,
        "section": section,
        "text": text,
        "length": len(text),
        "parent_id": parent_id,
    }

### 7.2 문장 단위 Chunking [선택: 원리 확인]

문장 부호(`.`, `!`, `?`) 뒤의 공백을 기준으로 나눈다.

In [24]:
# 마침표 자체는 보존하고, 문장 부호 바로 뒤의 공백만 분리 경계로 사용한다.
_SENTENCE_END = re.compile(r"(?<=[.!?])\s+")


def chunk_by_sentence(text: str, document_id: str) -> list[dict]:
    """문장 부호를 기준으로 문장 단위 Chunk를 만든다."""
    sentences = [s.strip() for s in _SENTENCE_END.split(text.strip()) if s.strip()]
    return [
        make_chunk(f"{document_id}-sent-{i}", document_id, None, s)
        for i, s in enumerate(sentences, start=1)
    ]


sentence_chunks = chunk_by_sentence(document_text, "report-01")
print(f"문장 단위 Chunk 수: {len(sentence_chunks)}")
print(sentence_chunks[0])

문장 단위 Chunk 수: 17
{'chunk_id': 'report-01-sent-1', 'document_id': 'report-01', 'section': None, 'text': '2026년 사내 리모트워크 운영 현황 보고서\n\n작성일: 2026-01-15\n작성부서: 인사운영팀\n문서구분: 내부 보고서\n\n1.', 'length': 70, 'parent_id': None}


**관찰**: 문서 앞부분(제목, 작성일, 작성부서 등 메타데이터)에는 마침표가 없어서,
첫 번째 실제 문장과 함께 하나의 Chunk로 뒤섞인다. 문장 부호만으로는 문서의 구조를
알 수 없다는 한계를 보여주는 예다.

### 7.3 고정 길이 Chunking [필수]

글자 수(`chunk_size`)를 기준으로 문서를 순서대로 자른다.

In [25]:
def chunk_by_fixed_length(text: str, document_id: str, chunk_size: int) -> list[dict]:
    """chunk_size 글자 단위로 문서를 순서대로 자른다."""
    if chunk_size <= 0:
        raise ValueError("chunk_size는 1 이상이어야 합니다.")
    text = text.strip()
    chunks = []
    # range의 start 값을 slice 시작점으로 사용하므로 겹침 없이 일정 간격으로 이동한다.
    for i, start in enumerate(range(0, len(text), chunk_size), start=1):
        piece = text[start:start + chunk_size]
        chunks.append(make_chunk(f"{document_id}-fixed-{i}", document_id, None, piece))
    return chunks


fixed_chunks = chunk_by_fixed_length(document_text, "report-01", chunk_size=150)
print(f"고정 길이(150자) Chunk 수: {len(fixed_chunks)}")
print(fixed_chunks[0])

고정 길이(150자) Chunk 수: 5
{'chunk_id': 'report-01-fixed-1', 'document_id': 'report-01', 'section': None, 'text': '2026년 사내 리모트워크 운영 현황 보고서\n\n작성일: 2026-01-15\n작성부서: 인사운영팀\n문서구분: 내부 보고서\n\n1. 개요\n본 보고서는 2025년 한 해 동안 시행된 리모트워크 제도의 운영 현황을 정리하고,\n2026년도 제도 개선 방향을 제시하기 위해 작성되었', 'length': 150, 'parent_id': None}


### 7.4 Overlap Chunking [필수]

**TODO**: `chunk_by_fixed_length_with_overlap()`을 작성한다.

- `overlap`은 `chunk_size`보다 작아야 한다. 그렇지 않으면 `ValueError`를 발생시킨다.
- 이동 거리(`step`)는 `chunk_size - overlap`이다.
- `start`를 0부터 `step`씩 늘려가며 `text[start:start + chunk_size]`를 자른다.
- 그 Chunk가 이미 문서 끝`(start + chunk_size >= len(text))`에 닿았다면 그 Chunk를 마지막으로 반복을 멈춘다. 
- 각 Chunk는 `make_chunk(f"{document_id}-overlap-{i}", document_id, None, piece)`로
  만든다. `i`는 1부터 시작한다.

예상 출력: `chunk_by_fixed_length_with_overlap("가나다라마바사", "d", chunk_size=4, overlap=1)`의
첫 번째 Chunk `text`는 `"가나다라"`, 두 번째 Chunk `text`는 `"라마바사"`이다(3글자씩
이동, 마지막 글자 `"라"`가 겹침).

In [26]:
def chunk_by_fixed_length_with_overlap(text: str, document_id: str, chunk_size: int, overlap: int) -> list[dict]:
    """chunk_size 글자 단위로 자르되, overlap 글자만큼 앞 Chunk와 겹치게 만든다."""
    if chunk_size <= 0:
        raise ValueError("chunk_size는 1 이상이어야 합니다.")
    if overlap < 0:
        raise ValueError("overlap은 0 이상이어야 합니다.")
    if overlap >= chunk_size:
        raise ValueError("overlap은 chunk_size보다 작아야 합니다.")
    text = text.strip()
    # 다음 시작점을 chunk_size가 아니라 차이만큼 옮겨 overlap 구간을 다시 포함한다.
    step = chunk_size - overlap
    chunks = []
    i = 1
    start = 0
    while start < len(text):
        piece = text[start:start + chunk_size]
        chunks.append(make_chunk(f"{document_id}-overlap-{i}", document_id, None, piece))
        # 마지막 Chunk가 문서 끝에 닿으면 중복된 짧은 꼬리 Chunk를 만들지 않고 종료한다.
        if start + chunk_size >= len(text):
            break
        start += step
        i += 1
    return chunks


demo_overlap = chunk_by_fixed_length_with_overlap("가나다라마바사", "d", chunk_size=4, overlap=1)
print([c["text"] for c in demo_overlap])

overlap_chunks = chunk_by_fixed_length_with_overlap(document_text, "report-01", chunk_size=150, overlap=30)
print(f"Overlap(150자, overlap 30) Chunk 수: {len(overlap_chunks)}")

['가나다라', '라마바사']
Overlap(150자, overlap 30) Chunk 수: 6


### 7.5 문단 단위 Chunking [선택: 원리 확인]

빈 줄(`\n\n`)을 기준으로 나눈다. 이 문서는 섹션 제목과 본문 사이에 빈 줄이 없어서,
문단 단위 결과는 "제목+본문"이 하나의 Chunk로 유지된다.

In [27]:
def chunk_by_paragraph(text: str, document_id: str) -> list[dict]:
    """빈 줄을 기준으로 문단 단위 Chunk를 만든다."""
    paragraphs = [p.strip() for p in text.strip().split("\n\n") if p.strip()]
    return [
        make_chunk(f"{document_id}-para-{i}", document_id, None, p)
        for i, p in enumerate(paragraphs, start=1)
    ]


paragraph_chunks = chunk_by_paragraph(document_text, "report-01")
print(f"문단 단위 Chunk 수: {len(paragraph_chunks)}")
print(paragraph_chunks[1]["text"])

문단 단위 Chunk 수: 8
작성일: 2026-01-15
작성부서: 인사운영팀
문서구분: 내부 보고서


### 7.6 구조 기반 Chunking [필수]

**TODO**: `chunk_by_structure()`를 작성한다. `"숫자. 제목"` 형태의 섹션 제목을 기준으로
제목과 본문을 하나의 섹션 Chunk로 만든다.

- `_HEADING_RE`로 문서 안의 모든 섹션 제목(`"1. 개요"` 형태)을 찾는다.
- 각 섹션의 범위는 그 제목의 시작 위치부터 다음 제목의 시작 위치 전까지(마지막
  섹션은 문서 끝까지)이다.
- 섹션 전체(제목+본문)를 하나의 Chunk로 만든다.
  - `chunk_id`는 `f"{document_id}-sec-{순번}"`, `section`은 제목이다.
  - 이 단계에서는 상위-하위 관계를 만들지 않으므로 `parent_id`는 `None`이다.

예상 출력: `chunk_by_structure(document_text, "report-01")`의 Chunk는 모두
`section` 값을 가지며, Chunk 개수는 문서의 섹션 개수(6개)와 같아야 한다.

In [28]:
_HEADING_RE = re.compile(r"^(\d+)\.\s+(.+)$", re.MULTILINE)


def chunk_by_structure(text: str, document_id: str) -> list[dict]:
    """"숫자. 제목" 섹션 구조를 기준으로 제목과 본문을 함께 나눈다."""
    text = text.strip("\n")
    # finditer 결과의 위치 정보를 이용해 각 제목부터 다음 제목 직전까지 자른다.
    headings = list(_HEADING_RE.finditer(text))
    chunks = []
    for idx, match in enumerate(headings):
        title = match.group(2).strip()
        section_start = match.start()
        # 마지막 섹션만 다음 제목이 없으므로 문서 끝을 경계로 사용한다.
        section_end = headings[idx + 1].start() if idx + 1 < len(headings) else len(text)
        section_text = text[section_start:section_end].strip()
        chunk_id = f"{document_id}-sec-{idx + 1}"
        chunks.append(make_chunk(chunk_id, document_id, title, section_text))
    return chunks


structure_chunks = chunk_by_structure(document_text, "report-01")
print(f"구조 기반 Chunk 수: {len(structure_chunks)}")
print(structure_chunks[2])

구조 기반 Chunk 수: 6
{'chunk_id': 'report-01-sec-3', 'document_id': 'report-01', 'section': '만족도 조사 결과', 'text': '3. 만족도 조사 결과\n전체 응답자의 74%가 리모트워크 제도에 만족한다고 응답하였다.\n불만족 사유로는 협업 지연(41%), 화상회의 피로도(27%), 장비 지원 부족(19%) 순으로 나타났다.', 'length': 108, 'parent_id': None}


### 7.7 `RecursiveCharacterTextSplitter` 적용 [필수]

직접 만든 함수는 원리를 이해하기 위한 예제다. 실제 Agentic RAG에서는 검증된
Text Splitter에 `chunk_size`, `chunk_overlap`, 구분자 우선순위를 지정해 사용한다.
다음 Notebook 03-2에 전달할 기본 Chunk는 이 결과를 사용한다.

In [29]:
# 앞쪽 구분자부터 시도하고 크기를 맞출 수 없을 때 더 작은 경계로 재귀적으로 내려간다.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""],
)
recursive_pieces = text_splitter.split_text(document_text)
rag_chunks = [
    make_chunk(f"report-01-rag-{i}", "report-01", None, piece)
    for i, piece in enumerate(recursive_pieces, start=1)
]
print(f"RAG 기본 Chunk 수: {len(rag_chunks)}")
print(rag_chunks[0])

RAG 기본 Chunk 수: 7
{'chunk_id': 'report-01-rag-1', 'document_id': 'report-01', 'section': None, 'text': '2026년 사내 리모트워크 운영 현황 보고서\n\n작성일: 2026-01-15\n작성부서: 인사운영팀\n문서구분: 내부 보고서', 'length': 66, 'parent_id': None}


### 7.8 Parent-Child Chunking [심화: 선택 실행]

검색은 작은 Child 단위로 정밀하게 수행하되, 답변 생성에는 상위 섹션 전체를
제공하고 싶을 때 Parent-Child 구조를 사용한다. Agent 기초 과정에서는 개념과
결과만 확인하고, 구현은 심화 학습으로 남긴다.

In [30]:
def build_parent_child_chunks(section_chunks: list[dict]) -> list[dict]:
    """섹션 Chunk를 Parent로 두고 문장 단위 Child Chunk를 추가한다."""
    # Parent도 결과에 유지하고, 작은 Child는 parent_id로 원래 섹션 맥락을 참조한다.
    chunks = list(section_chunks)
    for parent in section_chunks:
        body = parent["text"].split("\n", 1)[-1]
        sentences = [s.strip() for s in _SENTENCE_END.split(body) if s.strip()]
        for i, sentence in enumerate(sentences, start=1):
            chunks.append(
                make_chunk(
                    f"{parent['chunk_id']}-child-{i}",
                    parent["document_id"],
                    parent["section"],
                    f"{parent['section']}\n{sentence}",
                    parent_id=parent["chunk_id"],
                )
            )
    return chunks


parent_child_chunks = build_parent_child_chunks(structure_chunks)
child_chunks = [c for c in parent_child_chunks if c["parent_id"] is not None]
print(f"Parent {len(structure_chunks)}개 / Child {len(child_chunks)}개")
print(child_chunks[0])

Parent 6개 / Child 11개
{'chunk_id': 'report-01-sec-1-child-1', 'document_id': 'report-01', 'section': '개요', 'text': '개요\n본 보고서는 2025년 한 해 동안 시행된 리모트워크 제도의 운영 현황을 정리하고,\n2026년도 제도 개선 방향을 제시하기 위해 작성되었다.', 'length': 81, 'parent_id': 'report-01-sec-1'}


## 8. 실행 결과 관찰

필수 전략 네 가지를 같은 문서에 적용해 Chunk 수와 길이 통계를 비교한다.
문장·문단 직접 구현과 Parent-Child는 기본 비교에서 제외한다.

In [31]:
import pandas as pd

strategy_results = {
    "고정 길이(150자)": fixed_chunks,
    "Overlap(150자, overlap 30)": overlap_chunks,
    "구조 기반": structure_chunks,
    "RecursiveCharacterTextSplitter": rag_chunks,
}

summary_rows = []
for name, chunks in strategy_results.items():
    lengths = [c["length"] for c in chunks]
    summary_rows.append({
        "전략": name,
        "chunk_수": len(chunks),
        "평균_길이": round(sum(lengths) / len(lengths), 1),
        "최대_길이": max(lengths),
        "최소_길이": min(lengths),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,전략,chunk_수,평균_길이,최대_길이,최소_길이
0,고정 길이(150자),5,143.6,150,118
1,"Overlap(150자, overlap 30)",6,144.7,150,118
2,구조 기반,6,106.7,144,79
3,RecursiveCharacterTextSplitter,7,100.9,144,66


**결과 해석**: 구조 기반 Chunk만 `section` metadata를 채운다.
`RecursiveCharacterTextSplitter`는 문단·줄바꿈·문장·공백 순으로 가능한 경계를
찾아 자르므로 단순 고정 길이보다 실제 RAG에 바로 사용하기 좋다. `parent_id`는
필수 구조 기반 결과에서는 `None`이고, 7.8 심화 Parent-Child에서만 채워진다.

## 9. 핵심 비교 실험

### 9.1 Chunk Size 변경 [필수]

In [32]:
for size in [50, 150, 400]:
    chunks = chunk_by_fixed_length(document_text, "report-01", chunk_size=size)
    lengths = [c["length"] for c in chunks]
    print(f"chunk_size={size:4d} -> chunk 수={len(chunks):3d}, 평균 길이={sum(lengths) / len(lengths):.1f}")

chunk_size=  50 -> chunk 수= 15, 평균 길이=47.9
chunk_size= 150 -> chunk 수=  5, 평균 길이=143.6
chunk_size= 400 -> chunk 수=  2, 평균 길이=359.0


**관찰**: `chunk_size`가 작아질수록 Chunk 수는 늘어나고 평균 길이는 줄어든다.
`chunk_size`가 너무 작으면(예: 50) 하나의 완결된 정보가 여러 Chunk로 쪼개져 문맥이
분산된다.

### 9.2 Overlap 변경 [필수]

In [33]:
no_overlap = chunk_by_fixed_length_with_overlap(document_text, "report-01", chunk_size=100, overlap=0)
with_overlap = chunk_by_fixed_length_with_overlap(document_text, "report-01", chunk_size=100, overlap=30)
print(f"overlap=0  -> chunk 수={len(no_overlap)}")
print(f"overlap=30 -> chunk 수={len(with_overlap)}")

shared = with_overlap[0]["text"][-30:] == with_overlap[1]["text"][:30]
print("인접 Chunk가 30자를 공유하는가:", shared)

overlap=0  -> chunk 수=8
overlap=30 -> chunk 수=10
인접 Chunk가 30자를 공유하는가: True


**관찰**: Overlap을 늘리면 인접 Chunk끼리 내용을 공유해 경계에서 문맥이 끊기는
문제는 줄어들지만, Chunk 수가 늘어나 저장 공간과 검색 후보가 함께 증가한다.

### 9.3 제목과 본문 분리 문제 [필수]

In [34]:
heading = "3. 만족도 조사 결과"
section3_text = "3. 만족도 조사 결과\n전체 응답자의 74%가 리모트워크 제도에 만족한다고 응답하였다."

title_split = chunk_by_fixed_length(section3_text, "demo", chunk_size=len(heading))
for c in title_split:
    print(repr(c["text"]))

'3. 만족도 조사 결과'
'\n전체 응답자의 74%'
'가 리모트워크 제도에 '
'만족한다고 응답하였다.'


**관찰**: 고정 길이 Chunking은 글자 수만 계산하므로, 첫 번째 Chunk에는 제목만
남고 본문은 다음 Chunk로 분리된다. 제목이 없는 두 번째 Chunk만 검색되면, 이 내용이
"만족도 조사 결과"에 관한 것인지 알 수 없다.

### 9.4 문맥 단절 문제 [선택: 추가 관찰]

In [35]:
complaint_text = "불만족 사유로는 협업 지연(41%), 화상회의 피로도(27%), 장비 지원 부족(19%) 순으로 나타났다."
broken = chunk_by_fixed_length(complaint_text, "demo", chunk_size=20)
for c in broken:
    print(repr(c["text"]))

'불만족 사유로는 협업 지연(41%),'
' 화상회의 피로도(27%), 장비 지'
'원 부족(19%) 순으로 나타났다.'


**관찰**: `chunk_size=20`처럼 값이 작으면 "협업 지연(41%)"와 같은 하나의 항목이
Chunk 경계에서 잘린다. 각 Chunk만 놓고 보면 숫자와 항목명이 분리되어 의미를 알기
어렵다.

### 9.5 지나치게 큰 Chunk 문제 [선택: 추가 관찰]

In [36]:
huge_chunk = chunk_by_fixed_length(document_text, "report-01", chunk_size=len(document_text))[0]
headings_in_huge_chunk = _HEADING_RE.findall(huge_chunk["text"])
print(f"하나의 Chunk 안에 섞인 섹션 수: {len(headings_in_huge_chunk)}")

small_chunks = chunk_by_fixed_length(document_text, "report-01", chunk_size=150)
mixed_small_chunks = [c for c in small_chunks if len(_HEADING_RE.findall(c["text"])) > 1]
print(f"150자 Chunk 중 섹션이 2개 이상 섞인 Chunk 수: {len(mixed_small_chunks)}")

하나의 Chunk 안에 섞인 섹션 수: 6
150자 Chunk 중 섹션이 2개 이상 섞인 Chunk 수: 1


**관찰**: 문서 전체를 하나의 Chunk로 만들면 6개 섹션(운영 현황, 만족도, 문제점 등
서로 다른 주제)이 모두 한 Chunk에 섞인다. 이 Chunk가 검색되면 질문과 무관한 내용도
항상 함께 반환되어 검색 정밀도가 떨어진다.

## 10. 오류 수정 실습

9번에서 관찰한 "제목-본문 분리"와 "주제 혼합" 문제가 구조 기반 Chunking에서는
발생하지 않는지 재검증한다.

In [37]:
structure_chunks = chunk_by_structure(document_text, "report-01")
section_chunks = structure_chunks

# 상위 Chunk마다 섹션이 정확히 1개만 포함되는지 확인 (주제 혼합 문제 해결)
mixed_section_chunks = [c for c in section_chunks if len(_HEADING_RE.findall(c["text"])) != 1]
print(f"섹션이 1개가 아닌 구조 Chunk 수: {len(mixed_section_chunks)} (0이어야 정상)")

# 상위 Chunk의 section 값과 text 시작 부분이 같은 제목을 가리키는지 확인 (제목-본문 분리 문제 해결)
title_body_ok = all(c["text"].startswith(f"{i + 1}. {c['section']}") for i, c in enumerate(section_chunks))
print("모든 구조 Chunk가 제목으로 시작하는가:", title_body_ok)

섹션이 1개가 아닌 구조 Chunk 수: 0 (0이어야 정상)
모든 구조 Chunk가 제목으로 시작하는가: True


**결과 해석**: 구조 기반 Chunking은 섹션 경계를 알고 자르기 때문에 하나의 Chunk에
여러 주제가 섞이지 않고, 제목과 본문이 항상 함께 유지된다. 다만 이 방식은 문서에
`"숫자. 제목"` 같은 뚜렷한 구조가 있어야만 동작한다.

## 11. 도전 과제

1. `RecursiveCharacterTextSplitter`의 `chunk_size`와 `chunk_overlap`을 바꾸고
   Chunk 수와 경계가 어떻게 달라지는지 비교한다.
2. 구조 기반으로 먼저 섹션을 나눈 뒤, 각 섹션 안에서
   `RecursiveCharacterTextSplitter`를 적용하는 2단계 Chunking을 구현한다.
3. 두 번째 샘플 문서를 만들어 `document_id`를 다르게 지정하고, Notebook 03-2의
   metadata filter와 연결할 준비를 한다.

## 12. 테스트

**테스트 유형: 단위 테스트 — 결정적, 외부 API 호출 없음**

Chunk 크기·Overlap 검증과 Chunk metadata 구조를 확인한다.

In [38]:
for invalid_size in [0, -1]:
    try:
        chunk_by_fixed_length("abc", "t", chunk_size=invalid_size)
        raise AssertionError("chunk_size <= 0인데 예외가 발생하지 않았습니다.")
    except ValueError:
        pass

try:
    chunk_by_fixed_length_with_overlap("abc", "t", chunk_size=2, overlap=-1)
    raise AssertionError("overlap < 0인데 예외가 발생하지 않았습니다.")
except ValueError:
    pass

fixed_check = chunk_by_fixed_length("0123456789", "t", chunk_size=4)
assert [c["text"] for c in fixed_check] == ["0123", "4567", "89"]

try:
    chunk_by_fixed_length_with_overlap("0123456789", "t", chunk_size=4, overlap=4)
    raise AssertionError("overlap >= chunk_size인데 예외가 발생하지 않았습니다.")
except ValueError:
    pass

overlap_check = chunk_by_fixed_length_with_overlap("가나다라마바사", "t", chunk_size=4, overlap=1)
assert overlap_check[0]["text"] == "가나다라"
assert overlap_check[1]["text"][0] == overlap_check[0]["text"][-1]
assert len(overlap_check) == 2  # 끝부분만 중복되는 작은 꼬리 Chunk를 만들지 않는다.

structure_check = chunk_by_structure(document_text, "report-01")
assert len(structure_check) == 6
assert all(c["parent_id"] is None for c in structure_check)
assert all(c["section"] is not None for c in structure_check)

assert len(rag_chunks) > 0
assert all(0 < c["length"] <= 150 for c in rag_chunks)
assert all(c["document_id"] == "report-01" for c in rag_chunks)

for c in structure_check:
    assert set(c.keys()) == {"chunk_id", "document_id", "section", "text", "length", "parent_id"}

print("테스트 통과")

테스트 통과


## 13. 결과 저장

In [39]:
chunking_log = {
    "document": str(DATA_PATH.name),
    "document_length": len(document_text),
    "strategy_summary": summary_df.to_dict(orient="records"),
    "structure_chunks_sample": structure_chunks[:4],
    "rag_chunks_sample": rag_chunks[:4],
}
saved_path = save_log(chunking_log, OUTPUT_DIR / "logs" / "03-1_document_chunking_log.json")
print("저장 위치:", saved_path)

저장 위치: E:\agentic_ai_lab\outputs\logs\03-1_document_chunking_log.json


## 14. 핵심 정리

- Chunking은 문서를 검색과 LLM 입력에 적합한 크기로 나누는 전처리 과정이다.
- Chunk가 너무 작으면 문맥 손실, 너무 크면 여러 주제가 섞여 검색 정밀도가
  떨어진다.
- Overlap은 경계의 문맥 손실을 줄이지만 저장·검색 비용이 늘어나는 트레이드오프가
  있다.
- 구조 기반 Chunking은 제목-본문 연결과 `section` metadata를 보존한다.
- 문장·문단 직접 구현은 분리 원리를 확인하는 선택 학습이며, 실제 Agentic RAG는
  `RecursiveCharacterTextSplitter` 같은 기존 기능을 사용한다.
- Parent-Child는 작은 단위로 검색하고 큰 상위 맥락으로 답변할 때 사용하는 심화 전략이다.

## 15. 확인 문제

1. Chunk Size가 너무 작을 때와 너무 클 때 각각 어떤 문제가 발생하는가?
2. Overlap을 늘리면 무엇이 좋아지고, 대신 어떤 비용이 늘어나는가?
3. 직접 만든 고정 길이 함수 대신 `RecursiveCharacterTextSplitter`를 실제 RAG에서
   사용하는 이유는 무엇인가?
4. 구조 기반 Chunking이 보존하는 metadata는 무엇이며 검색에 어떻게 활용할 수 있는가?
5. Parent-Child Chunking은 어떤 경우에 필요한가?